In [ ]:
using Oceananigans
using Oceananigans.Units

using CairoMakie
using CUDA
using Printf
using Random
using SeawaterPolynomials.TEOS10: TEOS10EquationOfState

Random.seed!(1969) # for reproducible results

In [ ]:
#########CLOCK & ARCHITECHTURE############
clock = Clock(; time =0.0)
architecture = CPU()

############GRID DEFINITION###############

#Define number of cells in each direction
Cx = 50
Cy = 1
Cz = 40

#Define length of grid in each direction
Nx = 50 # Units: meters  3800m instead of 3700 because kelp has to be placed at a minimum 50m from the boundary(End of the line of rope)
Ny = 1 # Units: meters
Nz = 40  #Units: meters

#For 2D models, the y-axis is ignored completely and identified as flat.


#Bounded boundary conditions because the kelp forest is on the coast
grid = RectilinearGrid(size = (Cx, Cz), 
                          x =(0, Nx), 
                          y =(0, 1), 
                          z =(-Nz, 0),
                   topology = (Periodic, Flat, Bounded)) 

# #Bounded boundary conditions because the kelp forest is on the coast
# grid = RectilinearGrid(size = (Cx, Cy, Cz), 
#                           x =(0, Nx), 
#                           y =(0, Ny), 
#                           z =(-Nz, 0),
#                    topology = (Periodic, Flat, Bounded)) 

In [ ]:
u₁₀ = 5  # m s⁻¹, average wind velocity 10 meters above the ocean
cᴰ = 2e-3 # dimensionless drag coefficient
ρₐ = 1.2  # kg m⁻³, approximate average density of air at sea-level
ρₒ = 1e3
τx = - ρₐ / ρₒ * cᴰ * u₁₀ * abs(u₁₀) # m² s⁻²

# u_bcs = FieldBoundaryConditions(top = FluxBoundaryCondition(τx))
@inline bottom_drag(x, t, u) = - 1e-3 * u^2  # linear drag


u_bcs = FieldBoundaryConditions(top = FluxBoundaryCondition(τx),
                                # bottom = ValueBoundaryCondition(0))
                                bottom = FluxBoundaryCondition(bottom_drag, field_dependencies=:u))



In [ ]:

# Boundary condition functions for T and S (need x,t signature for 2D top BC)
@inline T_surface_bc(x, t) = 20 
@inline T_surface_bc(x, y, t) = T_surface_bc(x, t)

@inline S_surface_bc(x, t) = 34
@inline S_surface_bc(x, y, t) = S_surface_bc(x, t)


# Temperature & Salinity boundary conditions
T_bcs = FieldBoundaryConditions(
    top = ValueBoundaryCondition(T_surface_bc)
    #bottom = GradientBoundaryCondition(0.0)
)


evaporation_rate = 1e-3 / hour # m s⁻¹
@inline Jˢ(x, y, t, S, evaporation_rate) = - evaporation_rate * S # [salinity unit] m s⁻¹

@inline Jˢ(x, t, S, evaporation_rate) = - evaporation_rate * S # [salinity unit] m s⁻¹
evaporation_bc = FluxBoundaryCondition(Jˢ, field_dependencies=:S, parameters=evaporation_rate)
S_bcs = FieldBoundaryConditions(top=evaporation_bc)



boundary_conditions = ( T = T_bcs,
                        S = S_bcs,
                        u=u_bcs)


In [ ]:
using SeawaterPolynomials.TEOS10: TEOS10EquationOfState

teos10 = TEOS10EquationOfState()

buoyancy = SeawaterBuoyancy(equation_of_state=teos10)

In [ ]:
###############DEFINE MODEL################
# model configurations
model = NonhydrostaticModel(grid;
                            clock = clock,
                            advection = WENO(),
                            closure = SmagorinskyLilly(), #ScalarDiffusivity(ν=1e-5, κ=1e-2),
                            boundary_conditions = boundary_conditions,
                                coriolis = FPlane(f=1e-4),
                            # biogeochemistry = biogeochemistry,
                            tracers = (:T, :S)
                            # forcing = (u = u_velocity, w = w_velocity,),
                            # buoyancy = buoyancy
                            )      

########## Initial conditions for 2D #############
# @inline Tᵢ(x, z) = 20 * (z + 7) / 7
# @inline Tᵢ(x, y, z) = Tᵢ(x, z)
dTdz = 0.1 # K m⁻¹

# Temperature initial condition: a stable density gradient with random noise superposed.
Tᵢ(x, z) = 20 + dTdz * z + dTdz * model.grid.Lz * 2e-6 * Ξ(z)
Tᵢ(x,y, z) = 20 + dTdz * z + dTdz * model.grid.Lz * 2e-6 * Ξ(z)

@inline Sᵢ(x, z) = 34 
@inline Sᵢ(x, y, z) = Sᵢ(x, z)

Ξ(z) = randn() * z / model.grid.Lz * (1 + z / model.grid.Lz) # noise
uᵢ(x, z) = sqrt(abs(τx)) * 1e-3 * Ξ(z) 
uᵢ(x, y,z) = sqrt(abs(τx)) * 1e-3 * Ξ(z)

# #Units in mmol N/m3
set!(model, 
     T = Tᵢ, S = Sᵢ,u=uᵢ)
# #Define output writers:
T = model.tracers.T
S = model.tracers.S


# #Defining the velocities in the x, y and z directions
# u,w = model.velocities
# #Defining the vorticity by subtracting velocity in the x direction from velocity in the z direction since this is a 2D model
# ξ = ∂x(u) - ∂z(w)

In [ ]:
# #Units in mmol N/m3
# set!(model, 
#      T = Tᵢ, S = Sᵢ,u=uᵢ)
#      #DON = 3.0, DOC = 80.0


In [ ]:

# 1. Define the stop time for the simulation
stop_time = 24.0hours

# 2. Set up the TimeStepWizard for adaptive time-stepping
# This will try to keep the simulation stable while maximizing the time step.
#wizard = TimeStepWizard(cfl=0.7, max_change=1.1, max_Δt=1.2) # max_Δt is in seconds

# 3. Create the simulation with a starting Δt and the stop_time
simulation = Simulation(model, Δt=20, stop_time=stop_time)

# 4. Add the wizard to the simulation's callbacks
#simulation.callbacks[:wizard] = Callback(wizard, IterationInterval(100))

# 5. Define a progress message callback
progress(sim) = @info @printf("Iter: %d, time: %s, Δt: %s, wall time: %s",
                                iteration(sim),
                                prettytime(sim),
                                prettytime(sim.Δt),
                                prettytime(sim.run_wall_time))
simulation.callbacks[:progress] = Callback(progress, IterationInterval(100))

# 6. Update the output writer to save results once per day
simulation.output_writers[:jld2] = JLD2Writer(model,
                                (; T, S, ),
                                filename = "ocean_convection3.jld2",
                                schedule = TimeInterval(10minute),
                                overwrite_existing = true)

In [ ]:
conjure_time_step_wizard!(simulation, cfl=0.7)

In [ ]:
simulation.model.clock.time = 0.

In [ ]:
#Run the simulation
run!(simulation)

In [ ]:
filepath = "ocean_convection3.jld2"
Tt = FieldTimeSeries(filepath, "T")
St = FieldTimeSeries(filepath, "S")

times = Tt.times
n = Observable(length(times))

Tₙ = @lift interior(Tt[$n], :, 1, :)
Sₙ = @lift interior(St[$n], :, 1, :)

grid = Tt.grid
x = xnodes(grid, Center())
z = znodes(grid, Center())

title = @lift @sprintf("t = %s", prettytime(times[$n]))

axis_kwargs = (xlabel = "x (m)",
               ylabel = "z (m)",
               # aspect = AxisAspect(grid.Lx / grid.Lz),
               limits = ((0, grid.Lx), (-grid.Lz, 0)))

Tlims = (17., 20)
Slims = (33.95, 34.05)

fig = Figure(size = (900, 400), figure_padding = 5)

fig[0, 1:4] = Label(fig, title, fontsize=16, tellwidth=false)

ax_T = Axis(fig[1, 1]; title = "Temperature", axis_kwargs...)
ax_S = Axis(fig[1, 3]; title = "Salinity",    axis_kwargs...)

hm_T = heatmap!(ax_T, x, z, Tₙ; colormap = :thermal, colorrange = Tlims)
Colorbar(fig[1, 2], hm_T; label = "°C")

hm_S = heatmap!(ax_S, x, z, Sₙ; colormap = :haline, colorrange = Slims)
Colorbar(fig[1, 4], hm_S; label = "g / kg")

fig

In [ ]:
filename = "ocean_convection3"

intro = searchsortedfirst(times, 10minutes)
frames = intro:length(times)

@info "Making a motion picture..."
resize_to_layout!(fig)
CairoMakie.record(fig, filename * ".mp4", frames, framerate=8) do i
    n[] = i
end